In [53]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch version: 2.10.0+cu128
CUDA available: True
Device: cuda


In [54]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

In [55]:
DATA_PATH = "/kaggle/input/datasets/saur3x/wifi-sensing/synthetic_csi_data_1440min.parquet"

df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)
print(df.head())
print(df.columns.tolist())

Shape: (17280000, 33)
       amp_0      amp_1      amp_2      amp_3      amp_4      amp_5  \
0  14.916262  15.994662  26.353529  19.284451  15.961163  13.952287   
1  15.881388  18.913483  11.906879  22.146627  16.037645  13.877267   
2  11.911693  17.846769  11.775219  20.362837  15.570219  16.446939   
3  18.860847  13.144541  11.633047  11.439113  13.471443  16.640806   
4  14.159934  11.710629  23.099676  15.314356  15.919868  15.691349   

       amp_6      amp_7      amp_8      amp_9  ...   phase_8   phase_9  \
0  12.595302  21.101038  16.295670  10.296592  ... -2.284873  1.025342   
1  12.776961  13.013695  15.237158   9.497480  ...  2.011778  3.370583   
2  16.825188  15.872265  16.604788  21.467648  ... -2.127911  1.469524   
3  18.279400  15.447737  18.720516  17.896854  ... -3.207695 -3.000940   
4  23.473412  12.324613  14.208289  15.427694  ... -0.090546  0.622783   

   phase_10  phase_11  phase_12  phase_13  phase_14  occupancy  movement  \
0 -0.673811 -0.302334  1.58977

In [56]:
amp_cols = [f"amp_{i}" for i in range(15)]
phase_cols = [f"phase_{i}" for i in range(15)]

target_col = "heart_rate_bpm"

# occupancy input değil, sadece valid segment bulmak için kullanılacak
feature_cols = amp_cols + phase_cols + ["movement"]

required_cols = feature_cols + ["occupancy", target_col]
required_cols = list(dict.fromkeys(required_cols))

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Eksik kolonlar var: {missing_cols}")

df = df[required_cols].copy()
df = df.loc[:, ~df.columns.duplicated()].copy()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.ffill().bfill()

print("Original shape:", df.shape)
print("\nHR describe:")
print(df[target_col].describe())

print("\nZero HR count:", (df[target_col] == 0).sum())

print("\nOccupancy counts:")
print(df["occupancy"].value_counts())

valid_mask = (
    (df["occupancy"] == 1) &
    (df[target_col] > 30) &
    (df[target_col] < 180)
)

print("\nValid HR rows:", valid_mask.sum())
print("Invalid rows:", (~valid_mask).sum())

Original shape: (17280000, 33)

HR describe:
count    1.728000e+07
mean     5.547095e+01
std      3.708841e+01
min      0.000000e+00
25%      0.000000e+00
50%      7.320437e+01
75%      8.356441e+01
max      1.000000e+02
Name: heart_rate_bpm, dtype: float64

Zero HR count: 5256000

Occupancy counts:
occupancy
1    12024000
0     5256000
Name: count, dtype: int64

Valid HR rows: 12024000
Invalid rows: 5256000


In [57]:
phase_values = df[phase_cols].values.astype(np.float32)
phase_unwrapped = np.unwrap(phase_values, axis=0)

df[phase_cols] = phase_unwrapped

print(df[phase_cols].head())

    phase_0   phase_1   phase_2   phase_3   phase_4   phase_5   phase_6  \
0 -0.540846  0.205050 -2.488475 -4.786048  1.335560  0.502684 -1.245720   
1  1.973014  2.090393 -0.697770 -3.745276 -0.779445 -2.178476 -3.407807   
2  1.971437  1.329042 -3.478360 -0.785065 -2.045788 -3.961430 -4.109251   
3  0.826516  0.497963 -6.227783  0.586849 -0.003575 -6.247188 -5.173931   
4  0.703947 -1.733253 -3.344578  3.519156  2.660878 -6.857404 -3.947026   

    phase_7   phase_8   phase_9  phase_10  phase_11  phase_12  phase_13  \
0  3.770308 -2.284873  1.025342 -0.673811 -0.302334  1.589775  4.286218   
1  6.529910 -4.271408  3.370583 -0.699195  0.101232 -0.049195  1.733633   
2  5.373446 -2.127912  1.469524 -3.564596 -1.133533  0.802645  0.623177   
3  3.701398 -3.207696  3.282245 -4.596823 -1.637304  0.644784 -1.790346   
4  5.256572 -0.090547  0.622782 -6.899667 -1.959731  0.908690 -2.875048   

   phase_14  
0 -1.053540  
1 -0.414639  
2 -0.930371  
3 -2.343142  
4 -1.472830  


In [58]:
valid_array = valid_mask.values

segment_id = np.full(len(df), -1, dtype=np.int64)

current_segment = -1
in_segment = False

for i, is_valid in enumerate(valid_array):
    if is_valid:
        if not in_segment:
            current_segment += 1
            in_segment = True
        segment_id[i] = current_segment
    else:
        in_segment = False

df["segment_id"] = segment_id

segment_lengths = (
    df[df["segment_id"] >= 0]
    .groupby("segment_id")
    .size()
    .sort_values(ascending=False)
)

print("Number of valid segments:", len(segment_lengths))

print("\nTop segment lengths:")
print(segment_lengths.head(20))

print("\nSmallest segment lengths:")
print(segment_lengths.tail(20))

Number of valid segments: 102

Top segment lengths:
segment_id
38    396000
12    324000
69    324000
29    288000
22    288000
18    288000
75    288000
79    288000
81    288000
94    252000
24    252000
42    216000
25    216000
95    216000
91    216000
0     216000
92    216000
20    180000
32    180000
76    180000
dtype: int64

Smallest segment lengths:
segment_id
54     36000
45     36000
51     36000
52     36000
68     36000
62     36000
63     36000
61     36000
72     36000
70     36000
82     36000
78     36000
85     36000
83     36000
87     36000
89     36000
90     36000
86     36000
99     36000
101    36000
dtype: int64


In [59]:
WINDOW_SIZE = 512
STRIDE = 128

segment_lengths = df[df["segment_id"] >= 0].groupby("segment_id").size()

usable_segments = segment_lengths[segment_lengths >= WINDOW_SIZE].index.to_numpy()

rng = np.random.default_rng(SEED)
rng.shuffle(usable_segments)

num_segments = len(usable_segments)

train_segments = usable_segments[:int(num_segments * 0.70)]
val_segments = usable_segments[int(num_segments * 0.70):int(num_segments * 0.85)]
test_segments = usable_segments[int(num_segments * 0.85):]

print("Usable segments:", num_segments)
print("Train segments:", len(train_segments))
print("Val segments:", len(val_segments))
print("Test segments:", len(test_segments))

def describe_segments(name, selected_segments):
    mask = df["segment_id"].isin(selected_segments)
    print(f"\n{name}")
    print("Rows:", mask.sum())
    print(df.loc[mask, target_col].describe())

describe_segments("Train", train_segments)
describe_segments("Val", val_segments)
describe_segments("Test", test_segments)

Usable segments: 102
Train segments: 71
Val segments: 15
Test segments: 16

Train
Rows: 8640000
count    8.640000e+06
mean     7.918077e+01
std      9.966922e+00
min      6.000000e+01
25%      7.253316e+01
50%      7.875030e+01
75%      8.602652e+01
max      1.000000e+02
Name: heart_rate_bpm, dtype: float64

Val
Rows: 1404000
count    1.404000e+06
mean     8.103880e+01
std      1.082615e+01
min      6.000000e+01
25%      7.285074e+01
50%      7.838234e+01
75%      9.096951e+01
max      1.000000e+02
Name: heart_rate_bpm, dtype: float64

Test
Rows: 1980000
count    1.980000e+06
mean     8.113048e+01
std      1.167501e+01
min      6.000000e+01
25%      7.021905e+01
50%      8.238446e+01
75%      9.142571e+01
max      1.000000e+02
Name: heart_rate_bpm, dtype: float64


In [60]:
features_raw = df[feature_cols].values.astype(np.float32)
target_raw = df[target_col].values.astype(np.float32)
segments_raw = df["segment_id"].values

train_mask = df["segment_id"].isin(train_segments).values

scaler = StandardScaler()
scaler.fit(features_raw[train_mask])

features_scaled = scaler.transform(features_raw)

print("features_scaled:", features_scaled.shape)
print("target_raw:", target_raw.shape)
print("Feature count:", len(feature_cols))

features_scaled: (17280000, 31)
target_raw: (17280000,)
Feature count: 31


In [61]:
def make_windows_by_segments(
    features,
    target,
    segment_ids,
    selected_segments,
    window_size=256,
    stride=64
):
    X_windows = []
    y_windows = []

    for seg in selected_segments:
        idx = np.where(segment_ids == seg)[0]

        if len(idx) < window_size:
            continue

        seg_features = features[idx]
        seg_target = target[idx]

        for start in range(0, len(seg_features) - window_size + 1, stride):
            end = start + window_size

            X_windows.append(seg_features[start:end])
            y_windows.append(seg_target[start:end].mean())

    X_windows = np.array(X_windows, dtype=np.float32)
    y_windows = np.array(y_windows, dtype=np.float32)

    return X_windows, y_windows


X_train, y_train = make_windows_by_segments(
    features_scaled,
    target_raw,
    segments_raw,
    train_segments,
    WINDOW_SIZE,
    STRIDE
)

X_val, y_val = make_windows_by_segments(
    features_scaled,
    target_raw,
    segments_raw,
    val_segments,
    WINDOW_SIZE,
    STRIDE
)

X_test, y_test = make_windows_by_segments(
    features_scaled,
    target_raw,
    segments_raw,
    test_segments,
    WINDOW_SIZE,
    STRIDE
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

print("\nTrain target")
print("min:", y_train.min(), "max:", y_train.max(), "mean:", y_train.mean(), "std:", y_train.std())

print("\nVal target")
print("min:", y_val.min(), "max:", y_val.max(), "mean:", y_val.mean(), "std:", y_val.std())

print("\nTest target")
print("min:", y_test.min(), "max:", y_test.max(), "mean:", y_test.mean(), "std:", y_test.std())

X_train: (67264, 512, 31) y_train: (67264,)
X_val: (10918, 512, 31) y_val: (10918,)
X_test: (15415, 512, 31) y_test: (15415,)

Train target
min: 60.0 max: 100.0 mean: 79.17959 std: 9.606355

Val target
min: 60.125828 max: 100.0 mean: 81.04713 std: 10.82532

Test target
min: 60.0 max: 100.0 mean: 81.13116 std: 11.670471


In [62]:
def print_regression_results(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(name)
    print("-" * len(name))
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)
    print("Pred min:", y_pred.min())
    print("Pred max:", y_pred.max())
    print("Pred mean:", y_pred.mean())
    print("Pred std:", y_pred.std())
    print("True std:", y_true.std())
    print()

baseline_pred = np.full_like(y_test, y_train.mean())
print_regression_results("Baseline Test", y_test, baseline_pred)

Baseline Test
-------------
MAE : 10.427755355834961
RMSE: 11.832518715197867
R2  : -0.02796328067779541
Pred min: 79.17959
Pred max: 79.17959
Pred mean: 79.17958
Pred std: 7.6293945e-06
True std: 11.670471



In [63]:
def window_stats(X):
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    minv = X.min(axis=1)
    maxv = X.max(axis=1)

    return np.concatenate([mean, std, minv, maxv], axis=1)


X_train_stat = window_stats(X_train)
X_val_stat = window_stats(X_val)
X_test_stat = window_stats(X_test)

print("X_train_stat:", X_train_stat.shape)
print("X_val_stat:", X_val_stat.shape)
print("X_test_stat:", X_test_stat.shape)

X_train_stat: (67264, 124)
X_val_stat: (10918, 124)
X_test_stat: (15415, 124)


In [64]:
MAX_TRAIN_WINDOWS = 20_000

if len(X_train_stat) > MAX_TRAIN_WINDOWS:
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(len(X_train_stat), size=MAX_TRAIN_WINDOWS, replace=False)

    X_train_rf = X_train_stat[sample_idx]
    y_train_rf = y_train[sample_idx]
else:
    X_train_rf = X_train_stat
    y_train_rf = y_train

print("RF train subset:", X_train_rf.shape)

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

start_time = time.time()

rf.fit(X_train_rf, y_train_rf)

print("RF training time:", time.time() - start_time, "seconds")

rf_train_preds = rf.predict(X_train_stat)
rf_val_preds = rf.predict(X_val_stat)
rf_test_preds = rf.predict(X_test_stat)

print_regression_results("RandomForest Train", y_train, rf_train_preds)
print_regression_results("RandomForest Val", y_val, rf_val_preds)
print_regression_results("RandomForest Test", y_test, rf_test_preds)

baseline_pred = np.full_like(y_test, y_train.mean())
print_regression_results("Baseline Test", y_test, baseline_pred)

RF train subset: (20000, 124)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    9.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished


RF training time: 9.75865125656128 seconds
RandomForest Train
------------------
MAE : 0.36243453133693854
RMSE: 0.49690517313693977
R2  : 0.9973243476585787
Pred min: 60.142366001074066
Pred max: 99.8064770491391
Pred mean: 79.179913975002
Pred std: 9.516395063045472
True std: 9.606355

RandomForest Val
----------------
MAE : 3.918297106221806
RMSE: 5.589062567012961
R2  : 0.7334391484841263
Pred min: 66.12471232601852
Pred max: 96.75550765848011
Pred mean: 80.17589326052058
Pred std: 8.836625646169422
True std: 10.82532

RandomForest Test
-----------------
MAE : 4.494585479927749
RMSE: 5.3598384271302235
R2  : 0.7890757173287193
Pred min: 67.50129700153354
Pred max: 93.74765055199855
Pred mean: 81.0474784265971
Pred std: 8.023059882258753
True std: 11.670471

Baseline Test
-------------
MAE : 10.427755355834961
RMSE: 11.832518715197867
R2  : -0.02796328067779541
Pred min: 79.17959
Pred max: 79.17959
Pred mean: 79.17958
Pred std: 7.6293945e-06
True std: 11.670471



[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


In [65]:
try:
    from xgboost import XGBRegressor
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
except ImportError:
    raise ImportError("xgboost yüklü değil. Kaggle/Colab terminalde: pip install xgboost")


xgb_model = XGBRegressor(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    random_state=SEED,
    eval_metric="mae",
    early_stopping_rounds=50
)

start_time = time.time()

xgb_model.fit(
    X_train_stat,
    y_train,
    eval_set=[(X_val_stat, y_val)],
    verbose=50
)

print("XGBoost training time:", time.time() - start_time, "seconds")

xgb_train_preds = xgb_model.predict(X_train_stat)
xgb_val_preds = xgb_model.predict(X_val_stat)
xgb_test_preds = xgb_model.predict(X_test_stat)

print_regression_results("XGBoost Train", y_train, xgb_train_preds)
print_regression_results("XGBoost Val", y_val, xgb_val_preds)
print_regression_results("XGBoost Test", y_test, xgb_test_preds)

XGBoost version: 3.2.0
[0]	validation_0-mae:9.07430
[50]	validation_0-mae:4.28423
[100]	validation_0-mae:3.85748
[150]	validation_0-mae:3.83637
[174]	validation_0-mae:3.84030
XGBoost training time: 1.560237169265747 seconds
XGBoost Train
-------------
MAE : 1.5491811037063599
RMSE: 1.9665759359325388
R2  : 0.9580913186073303
Pred min: 62.30237
Pred max: 97.8828
Pred mean: 79.18035
Pred std: 8.7192955
True std: 9.606355

XGBoost Val
-----------
MAE : 3.821894884109497
RMSE: 4.905207237664235
R2  : 0.7946791052818298
Pred min: 66.55363
Pred max: 96.256516
Pred mean: 81.0746
Pred std: 9.093603
True std: 10.82532

XGBoost Test
------------
MAE : 4.7894721031188965
RMSE: 5.796931263941071
R2  : 0.7532714009284973
Pred min: 69.41831
Pred max: 93.62401
Pred mean: 81.58296
Pred std: 7.8973756
True std: 11.670471

